# Differential Item Functioning (DIF) with PCM --- Bayesian Estimation with Stan

## 학습 목표

1. DIF의 개념과 유형을 설명할 수 있다.
2. 한국어 평가에서 DIF가 발생하는 실제 상황을 예시로 들 수 있다.
3. Stan으로 PCM-DIF 모델을 추정하고 DIF 문항을 탐지할 수 있다.

---

## 1. DIF란 무엇인가? (What is DIF?)

**차별 문항 기능(Differential Item Functioning, DIF)**은 능력이 동일한 피험자 집단들이
특정 문항에서 체계적으로 다른 응답 패턴을 보이는 현상입니다.

$$\text{DIF} \Leftrightarrow P(X_i = k \mid \theta, G=1) \neq P(X_i = k \mid \theta, G=2)$$

- $G=1$: 기준 집단 (Reference Group)
- $G=2$: 초점 집단 (Focal Group)

> **핵심**: 능력($\theta$)을 통제한 후에도 집단 간 응답 확률 차이가 있을 때 DIF입니다.

---

## 2. DIF 유형 분류 (Categories of DIF)

### 2.1 통계적 분류

| 유형 | 설명 | 비고 |
|---|---|---|
| **균일 DIF (Uniform DIF)** | 모든 능력 수준에서 일관된 방향으로 한 집단이 유리함 | 난이도 차이 |
| **비균일 DIF (Non-uniform DIF)** | 능력 수준에 따라 방향이 달라짐 | 교차 패턴 |

### 2.2 원인에 따른 분류

| 분류 | 원인 | 예시 |
|---|---|---|
| **내용 DIF** | 문항 내용이 특정 집단에 친숙/낯섦 | 특정 국가 문화적 지식 요구 문항 |
| **언어 DIF** | 특정 언어 능력이 문항 이해에 영향 | 경어/존댓말 사용 맥락 |
| **방법 DIF** | 문항 형식이 특정 집단에 유리 | 듣기 vs 읽기 선호 차이 |
| **문화 DIF** | 문화적 배경 차이에 의한 응답 패턴 차이 | 한국 vs 해외 재외동포 |
| **젠더 DIF** | 성별에 따른 체계적 차이 | 특정 주제에 대한 친숙도 |
| **교육 DIF** | 학년/교육 배경 차이 | 교육 커리큘럼 차이 |

---

## 3. 한국어 평가에서 DIF의 실제 상황 (Real Examples)

### 3.1 TOPIK (한국어능력시험) DIF 사례

**사례 1: 문화 DIF**
- 문항: "다음 상황에서 적절한 표현을 고르시오 (명절 음식 준비)"
- 한국 거주 피험자(G=1): 문화적 맥락 친숙 → 더 쉽게 답변
- 해외 비거주 피험자(G=2): 문화적 맥락 낯섦 → 같은 언어 능력에도 더 어려움
- **이것이 DIF**: 언어 능력이 같아도 집단에 따라 정답률 차이 발생

**사례 2: 젠더 DIF**
- 문항: "다음 글에서 설명하는 직업은? (전통적 성별 직업 역할 맥락)"
- 여성 응시자(G=1): 해당 직업 관련 어휘에 더 친숙
- 남성 응시자(G=2): 같은 능력임에도 다른 응답 패턴
- **주의**: DIF ≠ 점수 차이. 집단 간 능력 차이(Impact)와 구별 필요

**사례 3: 쓰기 평가 DIF**
- 문항: "재외동포 경험에 대해 서술하시오"
- 재외동포(G=2): 직접 경험으로 유리
- 국내 학습자(G=1): 경험 없어 내용 구성 어려움
- 이것은 **채점 기준 개선 필요**를 시사

---

## 4. DIF 탐지 방법 비교

| 방법 | 특징 | 장단점 |
|---|---|---|
| **Mantel-Haenszel** | 비모수적, 오분류 행렬 기반 | 빠르지만 비균일 DIF 탐지 약함 |
| **Logistic Regression** | 능력×집단 상호작용 포함 | 비균일 DIF 탐지 가능 |
| **IRT-based (LR test)** | 파라미터 등가성 검정 | 이론적으로 강력 |
| **Bayesian PCM-DIF** (본 노트북) | 사후 분포 기반, 불확실성 정량화 | 소표본에서도 안정적 |


In [ ]:
import sys, os, warnings, tempfile
import matplotlib as mpl, matplotlib.font_manager as fm
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import cmdstanpy
from scipy import stats

warnings.filterwarnings('ignore')

def setup_korean_font():
    candidates = {
        'win32':  [('C:/Windows/Fonts/malgun.ttf', 'Malgun Gothic')],
        'darwin': [('/System/Library/Fonts/AppleSDGothicNeo.ttc', 'Apple SD Gothic Neo')],
        'linux':  [('/usr/share/fonts/truetype/nanum/NanumGothic.ttf', 'NanumGothic')],
    }
    for path, name in candidates.get(sys.platform, []):
        if os.path.exists(path):
            fm.fontManager.addfont(path)
            mpl.rcParams['font.family'] = [name, 'DejaVu Sans']
            return
    mpl.rcParams['font.family'] = ['DejaVu Sans']

setup_korean_font()
mpl.rcParams['axes.unicode_minus'] = False
np.random.seed(5101)


## 5. 시뮬레이션 설계

- **N = 200명** (기준 집단: 100명, 초점 집단: 100명)
- **I = 10 문항**, **K = 4 카테고리** (0-3점)
- **DIF 설정**: 문항 1, 2, 3에 DIF 있음; 나머지 문항은 DIF 없음
  - 문항 1: 초점 집단에게 더 어려움 (dif = +0.8)
  - 문항 2: 초점 집단에게 더 어려움 (dif = +0.5)
  - 문항 3: 초점 집단에게 더 쉬움 (dif = -0.6)
  - 문항 4-10: DIF 없음 (dif = 0)
- **group 변수**: 1=기준집단(국내 한국어 학습자), 2=초점집단(해외 재외동포)


In [ ]:
N1, N2 = 100, 100   # reference, focal groups
N  = N1 + N2
I  = 10
K  = 4

group = np.array([1]*N1 + [2]*N2)

# Both groups have similar ability distributions (important: DIF != Impact)
theta_true = np.concatenate([
    np.random.normal(0.0, 1.0, N1),   # reference group
    np.random.normal(0.1, 1.0, N2),   # focal group (slightly higher on average)
])

beta_raw  = np.random.normal(0, 0.8, I)
beta_true = beta_raw - beta_raw.mean()

# DIF effects: non-zero means group 2 (focal) has different item difficulty
dif_true = np.zeros(I)
dif_true[0] =  0.8   # Item 1: harder for focal group
dif_true[1] =  0.5   # Item 2: harder for focal group
dif_true[2] = -0.6   # Item 3: easier for focal group
# Items 4-10: no DIF

tau_raw  = np.random.normal(0, 0.7, (I, K-1))
tau_true = tau_raw - tau_raw.mean(axis=1, keepdims=True)

print("DIF simulation setup:")
print(f"  N_reference={N1}, N_focal={N2}, Items={I}, K={K}")
print("  DIF effects:")
for i in range(I):
    if dif_true[i] != 0:
        direction = 'harder for focal' if dif_true[i] > 0 else 'easier for focal'
        print(f"    Item {i+1}: dif={dif_true[i]:.1f} ({direction})")
    else:
        print(f"    Item {i+1}: no DIF")


In [ ]:
# Generate response data
def pcm_probs(theta, beta_eff, tau_i):
    log_p = np.zeros(K)
    for k in range(1, K):
        log_p[k] = log_p[k-1] + (theta - (beta_eff + tau_i[k-1]))
    log_p -= log_p.max()
    p = np.exp(log_p)
    return p / p.sum()

Y = np.zeros((N, I), dtype=int)
for n in range(N):
    g = group[n]
    for i in range(I):
        beta_eff = beta_true[i] + dif_true[i] * (1 if g == 2 else 0)
        pr = pcm_probs(theta_true[n], beta_eff, tau_true[i])
        Y[n, i] = np.random.choice(K, p=pr)

print(f"Response matrix Y: {Y.shape}")
print(f"Category counts: {np.bincount(Y.ravel())}")

# Visualize group-level item score distributions
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

item_means_g1 = Y[group==1].mean(axis=0)
item_means_g2 = Y[group==2].mean(axis=0)

x = np.arange(I)
width = 0.35
axes[0].bar(x - width/2, item_means_g1, width, label='Reference (Group 1)', color='steelblue', alpha=0.8)
axes[0].bar(x + width/2, item_means_g2, width, label='Focal (Group 2)', color='coral', alpha=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels([f'I{i+1}' for i in range(I)])
axes[0].set_ylabel('Mean Score')
axes[0].set_title('Mean Scores by Group (raw, not ability-controlled)')
axes[0].legend()

# DIF items highlighted
for i in range(I):
    if dif_true[i] != 0:
        axes[0].axvline(i, color='red', alpha=0.2, linewidth=8)

axes[1].hist(theta_true[group==1], bins=20, alpha=0.7, color='steelblue',
             label=f'Reference (mean={theta_true[group==1].mean():.2f})')
axes[1].hist(theta_true[group==2], bins=20, alpha=0.7, color='coral',
             label=f'Focal (mean={theta_true[group==2].mean():.2f})')
axes[1].set_xlabel('Ability theta')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Ability Distribution by Group\n(Similar distributions -> DIF != Impact)')
axes[1].legend()

fig.suptitle('DIF Simulation: Group Comparison', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()


## 6. Stan Model (PCM + DIF)

### DIF 모델링 전략

각 문항의 유효 난이도를 집단에 따라 다르게 설정:

```stan
real beta_eff = beta_base[i] + dif[i] * (group[n] == 2 ? 1 : 0);
```

- `beta_base[i]`: 기준 집단의 문항 난이도
- `dif[i]`: 초점 집단에서의 추가 난이도 (0 = no DIF)
- 규제화 사전 분포: `dif ~ normal(0, sigma_dif)`  
  → 데이터 근거 없이는 DIF를 0으로 수축 (Bayesian shrinkage)


In [ ]:
stan_code = 'data {\n  int<lower=1> N;\n  int<lower=1> I;\n  int<lower=2> K;\n  int<lower=0> Nobs;\n  array[Nobs] int<lower=1,upper=N> nn;\n  array[Nobs] int<lower=1,upper=I> ii;\n  array[Nobs] int<lower=1,upper=K> y;\n  array[N] int<lower=1,upper=2> group;   // 1 = reference, 2 = focal\n}\nparameters {\n  vector[N] theta;\n  real<lower=0> sigma_theta;\n\n  // Group-specific item difficulties: beta_ig = beta_base_i + dif_i * (g==2)\n  vector[I-1] beta_free;       // item difficulties (sum-to-zero), baseline (group 1)\n  vector[I] dif;               // DIF effect: 0 = no DIF; !=0 = DIF present\n  real<lower=0> sigma_dif;     // SD of DIF effects (shrinkage prior)\n\n  // Step difficulties (sum-to-zero per item for decomposed PCM)\n  matrix[I, K-2] tau_free;\n}\ntransformed parameters {\n  vector[I] beta_base;\n  matrix[I, K-1] tau;\n\n  beta_base = append_row(beta_free, -sum(beta_free));\n  for (i in 1:I)\n    tau[i] = append_col(tau_free[i], -sum(tau_free[i]));\n}\nmodel {\n  theta       ~ normal(0, sigma_theta);\n  sigma_theta ~ exponential(1);\n  beta_free   ~ normal(0, 2);\n  dif         ~ normal(0, sigma_dif);    // regularized DIF: shrink toward 0\n  sigma_dif   ~ exponential(2);         // encourage small DIF by default\n  to_vector(tau_free) ~ normal(0, 2);\n\n  for (obs in 1:Nobs) {\n    int n   = nn[obs];\n    int i   = ii[obs];\n    // Effective difficulty depends on group membership\n    real beta_eff = beta_base[i] + dif[i] * (group[n] == 2 ? 1 : 0);\n    vector[K] log_p;\n    log_p[1] = 0;\n    for (k in 2:K)\n      log_p[k] = log_p[k-1] + (theta[n] - (beta_eff + tau[i, k-1]));\n    y[obs] ~ categorical_logit(log_p);\n  }\n}\ngenerated quantities {\n  array[Nobs] int y_rep;\n  for (obs in 1:Nobs) {\n    int n   = nn[obs];\n    int i   = ii[obs];\n    real beta_eff = beta_base[i] + dif[i] * (group[n] == 2 ? 1 : 0);\n    vector[K] log_p;\n    log_p[1] = 0;\n    for (k in 2:K)\n      log_p[k] = log_p[k-1] + (theta[n] - (beta_eff + tau[i, k-1]));\n    y_rep[obs] = categorical_logit_rng(log_p);\n  }\n}'

nn_arr, ii_arr, y_arr = [], [], []
for n in range(N):
    for i in range(I):
        nn_arr.append(n + 1)
        ii_arr.append(i + 1)
        y_arr.append(int(Y[n, i]) + 1)

stan_data = {
    'N': N, 'I': I, 'K': K, 'Nobs': N * I,
    'nn': nn_arr, 'ii': ii_arr, 'y': y_arr,
    'group': group.tolist()
}

tmpdir = tempfile.mkdtemp()
stan_path = os.path.join(tmpdir, 'pcm_dif.stan')
with open(stan_path, 'w') as f:
    f.write(stan_code)

model = cmdstanpy.CmdStanModel(stan_file=stan_path)
print('PCM-DIF Stan model compiled.')


## 7. MAP Estimation

In [ ]:
fit_map = model.optimize(data=stan_data, seed=5101, jacobian=True)

dif_map       = fit_map.stan_variable('dif')
beta_base_map = fit_map.stan_variable('beta_base')

print("DIF MAP estimates vs True:")
for i in range(I):
    detected = 'DIF detected!' if abs(dif_map[i]) > 0.3 else 'no DIF'
    print(f"  Item {i+1}: true={dif_true[i]:+.2f}, MAP={dif_map[i]:+.3f}  -> {detected}")


## 8. Full Bayesian MCMC

In [ ]:
fit = model.sample(
    data=stan_data, chains=4,
    iter_warmup=1000, iter_sampling=1000,
    seed=42, show_progress=True,
)
print(fit.diagnose())


## 9. DIF Detection Results (사후 분포 기반 DIF 탐지)

In [ ]:
dif_samples = fit.stan_variable('dif')   # (4000, I)
dif_pm = dif_samples.mean(axis=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# DIF estimates with 96% CI
ax = axes[0]
dif_ci_lo = np.percentile(dif_samples, 2, axis=0)
dif_ci_hi = np.percentile(dif_samples, 98, axis=0)

colors = ['red' if (dif_ci_lo[i] > 0 or dif_ci_hi[i] < 0) else 'steelblue'
          for i in range(I)]

for i in range(I):
    ax.plot([dif_ci_lo[i], dif_ci_hi[i]], [i, i], 'k-', linewidth=1.5)
    ax.scatter(dif_pm[i], i, color=colors[i], s=80, zorder=5)
    ax.scatter(dif_true[i], i, marker='^', color='green', s=80, zorder=10)

ax.axvline(0, color='gray', linestyle='--', linewidth=1.5)
ax.set_yticks(range(I))
ax.set_yticklabels([f'Item {i+1}' for i in range(I)])
ax.set_xlabel('DIF effect (logit)\n0 = no DIF; positive = harder for focal group')
ax.set_title('Posterior Mean DIF with 96% CI\n(red = significant DIF, green triangle = true)')
ax.text(0.6, 0.98, 'Red = CI excludes 0\n(statistically significant DIF)',
        transform=ax.transAxes, fontsize=8, va='top',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

# DIF True vs Posterior Mean scatter
ax = axes[1]
ax.scatter(dif_true, dif_pm, color=['red' if abs(d) > 0 else 'steelblue' for d in dif_true],
           s=80, zorder=5, alpha=0.9)
lo = min(dif_true.min(), dif_pm.min()) - 0.2
hi = max(dif_true.max(), dif_pm.max()) + 0.2
ax.plot([lo, hi], [lo, hi], 'k--', linewidth=1.2, label='y=x')
ax.axhline(0, color='gray', linestyle=':', alpha=0.5)
ax.axvline(0, color='gray', linestyle=':', alpha=0.5)
r = np.corrcoef(dif_true, dif_pm)[0, 1]
for i in range(I):
    ax.text(dif_true[i]+0.03, dif_pm[i], f'I{i+1}', fontsize=8)
ax.set_xlabel('True DIF effect')
ax.set_ylabel('Posterior mean DIF')
ax.set_title(f'DIF Recovery  r={r:.3f}')
ax.legend()

fig.suptitle('DIF Detection Results --- PCM Bayesian Approach', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print("\n=== DIF Detection Summary ===")
print(f"{'Item':>6} {'True DIF':>10} {'Post.Mean':>10} {'96% CI':>20} {'Significant':>12}")
for i in range(I):
    sig = 'YES ***' if (dif_ci_lo[i] > 0 or dif_ci_hi[i] < 0) else 'no'
    print(f"  I{i+1:2d}  {dif_true[i]:>10.3f} {dif_pm[i]:>10.3f}  "
          f"[{dif_ci_lo[i]:+.3f}, {dif_ci_hi[i]:+.3f}]  {sig:>12}")


## 10. Convergence Analysis

In [ ]:
summary = fit.summary()
theta_s = summary[summary.index.str.startswith('theta[')]
dif_s   = summary[summary.index.str.startswith('dif[')]

print("Convergence:")
for nm, df in [('theta', theta_s), ('dif', dif_s)]:
    ess_col = 'N_Eff' if 'N_Eff' in df.columns else 'ESS_bulk'
    print(f"  {nm:5s}: Rhat max={df['R_hat'].max():.4f}, ESS min={df[ess_col].min():.0f}")

dif_samp = fit.stan_variable('dif')
fig, axes = plt.subplots(3, 2, figsize=(13, 9))
chain_colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
dif_items = [0, 1, 2]   # DIF items

for row, i in enumerate(dif_items):
    samp = dif_samp[:, i]
    chains = samp.reshape(4, 1000)
    for c in range(4):
        axes[row, 0].plot(range(1000), chains[c], color=chain_colors[c],
                          alpha=0.7, linewidth=0.6, label=f'C{c+1}' if row==0 else '')
    axes[row, 0].set_title(f'Trace: DIF_item_{i+1} (true={dif_true[i]:.2f})', fontsize=10)
    if row==0: axes[row, 0].legend(fontsize=7)
    axes[row, 1].hist(samp, bins=40, density=True, color='coral', alpha=0.6, edgecolor='white')
    axes[row, 1].axvline(samp.mean(), color='red', linestyle='--', linewidth=1.5)
    axes[row, 1].axvline(dif_true[i], color='green', linewidth=2.2,
                          label=f'True={dif_true[i]:.2f}')
    axes[row, 1].axvline(0, color='gray', linestyle=':', alpha=0.5)
    axes[row, 1].set_title(f'Posterior: DIF_item_{i+1}', fontsize=10)
    axes[row, 1].legend(fontsize=8)

axes[-1, 0].set_xlabel('Iteration')
fig.suptitle('Trace Plots: DIF Items (Items 1, 2, 3)', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()


## 11. PPC and Scatter Plots

In [ ]:
y_rep_all  = fit.stan_variable('y_rep')
theta_samp = fit.stan_variable('theta')
theta_pm   = theta_samp.mean(axis=0)

y_flat = np.array(y_arr) - 1
rep_means = (y_rep_all - 1).mean(axis=1)
obs_mean  = y_flat.mean()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].hist(rep_means, bins=40, density=True, color='steelblue', alpha=0.7,
             edgecolor='white', label='y_rep')
axes[0].axvline(obs_mean, color='red', linewidth=2.5, label=f'Obs={obs_mean:.3f}')
axes[0].set_title('PPC: Mean Score'); axes[0].legend()

axes[1].scatter(theta_true, theta_pm, color=['steelblue' if g==1 else 'coral' for g in group],
                s=20, alpha=0.7)
lims = [-3.5, 3.5]
axes[1].plot(lims, lims, 'k--')
axes[1].set_xlim(lims); axes[1].set_ylim(lims)
r = np.corrcoef(theta_true, theta_pm)[0,1]
axes[1].set_xlabel('True theta'); axes[1].set_ylabel('Posterior mean theta')
axes[1].set_title(f'Person Ability Recovery  r={r:.3f}')
axes[1].text(0.05, 0.92, 'blue=reference, red=focal', transform=axes[1].transAxes, fontsize=8)

plt.tight_layout(); plt.show()


## 12. Uncertainty Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
dif_ci_w = np.percentile(dif_samp, 98, axis=0) - np.percentile(dif_samp, 2, axis=0)
colors = ['red' if abs(dif_true[i]) > 0 else 'steelblue' for i in range(I)]
ax.bar(range(I), dif_ci_w, color=colors, alpha=0.75, edgecolor='white')
ax.axhline(dif_ci_w.mean(), color='black', linestyle='--', linewidth=1.5,
           label=f'Mean width = {dif_ci_w.mean():.3f}')
ax.set_xticks(range(I))
ax.set_xticklabels([f'I{i+1}' for i in range(I)])
ax.set_xlabel('Item')
ax.set_ylabel('96% CI Width (logit)')
ax.set_title('DIF Uncertainty: 96% Posterior CI Widths\n(red = items with true DIF)')
ax.legend()
plt.tight_layout(); plt.show()


## 13. References / 참고 문헌### 국제 학술지 (확인된 논문)1. **Joo, S.-H., Lee, P., & Stark, S. (2022)**. Bayesian Approaches for Detecting Differential Item Functioning Using the Generalized Graded Unfolding Model. *Applied Psychological Measurement, 46*(2), 98–115. https://doi.org/10.1177/014662162110666062. **Bürkner, P.-C. (2021)**. Bayesian Item Response Modeling in R with brms and Stan. *Journal of Statistical Software, 100*(5), 1–54. https://doi.org/10.18637/jss.v100.i053. **Masters, G. N. (1982)**. A Rasch model for partial credit scoring. *Psychometrika, 47*(2), 149–174. *(PCM 원전)*4. **Luo, Y., & Jiao, H. (2018)**. Using the Stan Program for Bayesian Item Response Theory. *Educational and Psychological Measurement, 78*(3), 384–408.5. **Gelman, A., Vehtari, A., Simpson, D., Margossian, C. C., Carpenter, B., Yao, Y., Kennedy, L., Gabry, J., Bürkner, P.-C., & Modrák, M. (2020)**. Bayesian Workflow. *arXiv:2011.01808*.---### KCI 등재 학술지6. **Rasch 모형을 적용한 문항분석 및 차별기능문항 탐색 — 2021학년도 물리인증제 1-2급을 바탕으로** (KCI 논문번호: ART002889793). KCI 포털(www.kci.go.kr)에서 전문 검색 가능.7. **문항반응이론을 활용한 학문 목적 한국어 말하기 평가 문항 및 채점자 특성 분석** (KCI 논문번호: ART002419170). KCI 포털에서 검색 가능.8. **한국어 학습자의 쓰기 수행 평가 신뢰도 분석 — 다국면 라쉬 모형을 사용하여** (KCI 논문번호: ART002387352). KCI 포털에서 검색 가능.---> **KCI 검색 방법**: www.kci.go.kr → 논문검색> 추천 검색어: `차별 문항 기능`, `DIF`, `문항 편파성`, `TOPIK 공정성`, `측정 동등성`